***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [9. 实践部分](9_1_visualisation-inspection.ipynb)
    * 上一节：[9.37 真实 PyBDSF 产品复盘](9_37_pybdsf_real_product_replay.ipynb)
    * 下一节：[9.y 综合实践问题集](9_problem_set.ipynb)

***


## 9.38 BIMA NGC 4826 Measurement Set 校准复盘

本节使用 casacore 官方仓库的 field-selection 测试 Measurement Set。TaQL 将其识别为 BIMA/MIRIAD 观测：主表有 14,985 行、10 根天线、6 个 data descriptions，包含 3C273 校准场和 NGC 4826 七场 mosaic；观测日期为 1998-04-16，项目标识为 `t108c115.n48`。原始 `.ms.tgz` 被原样保留，并通过 `extract_visibility.py` 导出两个轻量 NumPy 视图，使教材的默认 Python 3.10+ 环境可以直接复盘 `DATA`、`FLAG`、`UVW`、`WEIGHT`、天线和时间列。

本案例跨过了上一节“只有图像产品”的边界：学生将直接检查可见度、flag 和 $uv$ 覆盖，并在 3C273 数据上求解标量复增益。但它仍然是 casacore 测试夹具，不是完整归档交付：未找到原始归档编号、完整观测日志和可独立核对的 3C273 通量模型。因此校准只在“单位点源模型”下讨论相对增益，不建立绝对通量标度。


### 9.38.1 文件身份与 MS 契约

样本包位于 `sample_packages/bima_ngc4826_ms_replay/`。manifest 记录 casacore 源 commit、测试 MS 首次加入的历史 commit、上游 LGPLv2 许可证和逐文件 SHA-256。压缩包中保留主表和 `ANTENNA`、`FIELD`、`SPECTRAL_WINDOW`、`POLARIZATION`、`DATA_DESCRIPTION`、`SOURCE`、`SYSCAL`、`OBSERVATION`、`HISTORY` 等标准子表；`derived/ms_metadata.yaml` 固定保存与教学判断有关的 TaQL 证据摘要。本仓库不在工作区直接展开 MS，避免把 casacore `table.lock` 和平台相关的表状态当成可编辑源文件。


In [ ]:
import importlib.util
from pathlib import Path

package_dir = Path('sample_packages/bima_ngc4826_ms_replay')
if not package_dir.exists():
    package_dir = Path('9_Practical') / package_dir
spec = importlib.util.spec_from_file_location('bima_ms_replay', package_dir / 'analyze_ms.py')
replay = importlib.util.module_from_spec(spec)
spec.loader.exec_module(replay)
visibility_summary = replay.visibility_summary()
visibility_summary


### 9.38.1a 绝对标度证据矩阵

SOURCE 表确认 3C273、1310+323 和 NGC 4826 身份，并给出 NGC 4826 谱线静止频率，但没有通量列。SYSCAL 表有 20 行 `TSYS`、两组时间样本和固定的 `BIMA_JYPERK=140`；HISTORY 记录 `CPASSCAL=Y`、`CAPLCAL=Y`、11.5 s 积分以及 MIRIAD 到 MS2 的转换。主表 `DATA` 列却没有 `QuantumUnits`。这些记录证明数据经过上游校正流程并保留系统量，不足以独立重建 3C273 在该历元和频率的通量模型，也不足以声明当前数值单位为 Jy。


In [ ]:
metadata_evidence = replay.metadata_summary()
scale_evidence = metadata_evidence['absolute_flux_scale']
print('Sources:', [record['name'] for record in metadata_evidence['source_table']['records']])
print('TSYS range [K]:', metadata_evidence['syscal']['tsys_k_range'])
print('BIMA JY/K:', metadata_evidence['syscal']['bima_jyperk_values'])
for key, value in scale_evidence.items():
    print(f'{key}: {value}')


### 9.38.2 Flag 分布不只是一个百分比

3C273 提取包有 2,925 行、65 个时间样本和单通道数据，总 flag 比例约 47.8%。这个数字看似很高，但它主要来自部分天线在该选择中完全不可用，而不是所有基线均匀随机损失一半。对求解器而言，后者可能仅降低 S/N，前者则会改变天线约束图的连通性。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

with np.load(package_dir / 'derived/calibrator_3c273_ddid0.npz') as archive:
    calibrator = {key: archive[key] for key in archive.files}

times, inverse = np.unique(calibrator['time_s'], return_inverse=True)
median_amplitude = np.array([
    np.median(np.abs(calibrator['data'][inverse == index][~calibrator['flag'][inverse == index]]))
    for index in range(len(times))
])
antennas = np.unique(np.r_[calibrator['antenna1'], calibrator['antenna2']])
flag_by_antenna = []
for antenna in antennas:
    rows = (calibrator['antenna1'] == antenna) | (calibrator['antenna2'] == antenna)
    flag_by_antenna.append(np.mean(calibrator['flag'][rows]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot((times - times.min()) / 3600, median_amplitude, marker='o', markersize=3)
axes[0].set_xlabel('Time since start [h]')
axes[0].set_ylabel('Median unflagged amplitude [native unit]')
axes[0].grid(alpha=0.3)
axes[1].bar(antennas, flag_by_antenna)
axes[1].set_xlabel('Antenna ID')
axes[1].set_ylabel('Flag fraction')
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


### 9.38.3 在真实可见度上求相对增益

前 42 个时间样本中，从参考天线 0 出发的非零权重图只连通 7 根天线；后续变为 8 根。为了使所有 65 个独立时间解都有相同的参数空间，本实验只使用全程连通且无 flag 的天线 `[0, 1, 2, 3, 4, 6, 7]`。对相位中心的未分辨校准源，将模型归一化为 $m_{pq}=1$，再求解

$$d_{pq}=g_p g_q^* m_{pq}+n_{pq}.$$

因为模型通量被设为 1，增益幅度吸收了未知的校准源通量尺度。校正后可见度接近单位点源，只证明天线基标量模型能解释大部分数据；它不能把结果转换为绝对 Jy 标度。


In [ ]:
calibration = replay.calibration_summary()
gains = calibration['gains']
solution_time = (times - times.min()) / 3600

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for column, antenna in enumerate(calibration['active_antennas']):
    axes[0].plot(solution_time, abs(gains[:, column]), label=str(antenna))
    axes[1].plot(solution_time, np.angle(gains[:, column]), label=str(antenna))
axes[0].set_ylabel('Relative gain amplitude')
axes[1].set_ylabel('Gain phase [rad]')
axes[1].set_xlabel('Time since start [h]')
for ax in axes:
    ax.grid(alpha=0.3)
axes[0].legend(title='Antenna', ncol=4)
plt.tight_layout()
plt.show()

print(f"Normalized residual RMS before solve: {calibration['normalized_rms_before']:.3f}")
print(f"Normalized residual RMS after solve:  {calibration['normalized_rms_after']:.3f}")
print(f"Residual ratio: {calibration['rms_ratio']:.3%}")


### 9.38.3a 增益表、解区间与基线留出验证

每个时间样本的 7 根活动天线形成完整的 21 条基线。下面固定留出 7 条基线，只用其余 14 条连通基线求解，再在从未进入拟合的基线上计算残差。这样可以区分训练残差下降与跨基线预测能力。样本包中的 `relative_gain_table.npz` 记录时间、真实天线 ID、复增益、参考天线、输入支持权重、flag、训练/留出基线和逐时间残差；输入支持权重是相连基线权重之和，不是增益形式协方差。最终增益由全部 21 条基线重算，留出解只承担 QA。

单积分解的训练 RMS 为约 0.138、留出 RMS 为约 0.233；把 5 个积分合成一个解后，留出 RMS 升至约 0.385，整段扫描只求一个解时约为 0.398。这个结果支持本扫描存在不能被长解区间跟踪的时间变化，但不自动证明 12 秒解适合转移到目标场：校准源模型仍无绝对通量标尺，目标与校准源之间还有约 989 秒间隔。


In [ ]:
with np.load(package_dir / 'derived/relative_gain_table.npz') as archive:
    gain_table = {key: archive[key] for key in archive.files}
interval_qa = replay.calibration_interval_summary()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(solution_time, gain_table['training_rms'], label='Training')
axes[0].plot(solution_time, gain_table['holdout_rms'], label='Held out')
axes[0].set_xlabel('Time since start [h]')
axes[0].set_ylabel('Normalized residual RMS')
axes[0].legend()
labels = [str(item['interval_samples']) for item in interval_qa]
axes[1].plot(labels, [item['training_rms'] for item in interval_qa], marker='o', label='Training')
axes[1].plot(labels, [item['holdout_rms'] for item in interval_qa], marker='o', label='Held out')
axes[1].set_xlabel('Integrations per solution')
axes[1].set_ylabel('Normalized residual RMS')
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Held-out baselines:', gain_table['holdout_baseline_pairs'].tolist())
print(interval_qa)


### 9.38.4 目标场的频谱和 $uv$ 覆盖

NGC 4826 提取包选择 mosaic 的一个 field 和第一个 64 通道谱窗，共 360 行。这个小切片适合检查两类处理前事实：频谱结构是否支持平均或连续谱扣除，以及 $uv$ 距离范围是否支持所需角尺度。它不包含全部七个 mosaic fields 和全部谱窗，因此下图不是最终联合成像的完整覆盖。


In [ ]:
with np.load(package_dir / 'derived/target_ngc4826_field2_ddid2.npz') as archive:
    target = {key: archive[key] for key in archive.files}
frequency_hz = np.load(package_dir / 'derived/target_ddid2_channel_frequency_hz.npy')
masked_amplitude = np.ma.array(np.abs(target['data']), mask=target['flag'])
median_spectrum = np.ma.median(masked_amplitude, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(frequency_hz / 1e9, median_spectrum)
axes[0].set_xlabel('Frequency [GHz]')
axes[0].set_ylabel('Median visibility amplitude [native unit]')
axes[0].grid(alpha=0.3)
axes[1].scatter(target['uvw_m'][:, 0], target['uvw_m'][:, 1], s=8, alpha=0.5)
axes[1].scatter(-target['uvw_m'][:, 0], -target['uvw_m'][:, 1], s=8, alpha=0.5)
axes[1].set_xlabel('u [m]')
axes[1].set_ylabel('v [m]')
axes[1].set_aspect('equal', adjustable='box')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 9.38.4a 完整目标 mosaic 的 field/DDID 覆盖

单一 field/DDID 只是复盘切片。样本包还保留了所有 NGC 4826 目标 field 和 4 个 64 通道数据描述的 UVW 覆盖，可用来检查合并成像之前的现实 field/DDID 分布。覆盖提取显示，校准场结束到目标场开始相隔约 989 s；因此本节不把 3C273 的相对增益直接同步转移到 mosaic。若要这样做，至少还需要时间相关的增益模型、校准源与目标的观测顺序及完整观测元数据。


In [ ]:
coverage_summary = replay.coverage_summary()
with np.load(package_dir / 'derived/target_ngc4826_mosaic_coverage.npz') as archive:
    coverage = {key: archive[key] for key in archive.files}

fig, ax = plt.subplots(figsize=(6, 5))
for field in np.unique(coverage['field_id']):
    selected = coverage['field_id'] == field
    ax.scatter(
        coverage['uvw_m'][selected, 0],
        coverage['uvw_m'][selected, 1],
        s=5,
        alpha=0.35,
        label=f'Field {field}',
    )
ax.set_xlabel('u [m]')
ax.set_ylabel('v [m]')
ax.set_aspect('equal', adjustable='box')
ax.grid(alpha=0.3)
ax.legend(ncol=2, fontsize='small')
plt.tight_layout()
plt.show()
print(coverage_summary)


### 9.38.5 从真实可见度到 dirty image

下面的最小成像器只使用一个 NGC 4826 field 和一个 DDID：先对未 flag 的通道做平均，再加入共轭 $uv$ 点，用 nearest-neighbour 网格化和二维逆 FFT 生成 dirty image 与 PSF。它没有使用完整 mosaic、精确 WCS、主波束、复杂加权或 w-term 修正，因此是成像前后链路的透明基线，不是 CASA/WSClean 的替代品。


In [ ]:
valid = ~target['flag']
channel_count = valid.sum(axis=1)
rows = channel_count > 0
visibility = (target['data'] * valid).sum(axis=1)[rows] / channel_count[rows]
frequency = frequency_hz.mean()
wavelength = 299792458.0 / frequency
u = target['uvw_m'][rows, 0] / wavelength
v = target['uvw_m'][rows, 1] / wavelength

npix = 256
umax = max(np.max(abs(u)), np.max(abs(v)))
du = 2 * umax / (npix * 0.8)
cell_arcsec = 206265.0 / (npix * du)
uv_grid = np.zeros((npix, npix), complex)
weight_grid = np.zeros((npix, npix), float)
for uu, vv, sample in zip(
    np.r_[u, -u], np.r_[v, -v], np.r_[visibility, visibility.conj()]
):
    ix = int(round(uu / du)) + npix // 2
    iy = int(round(vv / du)) + npix // 2
    if 0 <= ix < npix and 0 <= iy < npix:
        uv_grid[iy, ix] += sample
        weight_grid[iy, ix] += 1.0

dirty = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(uv_grid))).real
psf = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(weight_grid))).real
dirty /= psf.max()
psf /= psf.max()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
dirty_limit = np.percentile(abs(dirty), 99.5)
axes[0].imshow(dirty, origin='lower', cmap='coolwarm', vmin=-dirty_limit, vmax=dirty_limit)
axes[0].set_title('Dirty image from one field/DDID')
axes[1].imshow(psf, origin='lower', cmap='coolwarm', vmin=-0.2, vmax=1.0)
axes[1].set_title('Nearest-neighbour PSF')
for ax in axes:
    ax.set_xlabel(f'Pixel; cell={cell_arcsec:.2f} arcsec')
    ax.set_ylabel('Pixel')
plt.tight_layout()
plt.show()
print(f'Unflagged rows: {len(visibility)}; grid samples: {np.count_nonzero(weight_grid)}')
print(f'Approximate cell size: {cell_arcsec:.2f} arcsec')


图像和 PSF 的存在只证明可见度已经通过一个透明的数值网格化流程。网格孔径和图像周期边界会改变旁瓣；未校准数据会改变峰值和残差；未加入其他 field/DDID 则不能代表完整 NGC 4826 mosaic。这些是应在主要科学测量之前写进报告的限制，不是可以用一个更大 `imsize` 消除的技术细节。


### 9.38.6 定量任务（35 分）

1. 从 flag-by-antenna 图重建每个时段的天线约束图，证明为什么“总 flag 比例 47.8%”不足以判断求解是否可行。（6 分）
2. 复现固定基线留出实验，比较 1、5、65 个积分解区间的参数数量、训练残差、留出残差和解连续性；解释为什么最终表应在选定解区间后用全部基线重算。（6 分）
3. 将参考天线从 0 改为 2，验证增益相位发生规范变换，而校正可见度和残差不变。（4 分）
4. 从目标提取包计算每通道的 flag 比例和复振幅分布，识别不应在成像前盲目平均的频率结构。（5 分）
5. 用中心频率把 $uv$ 距离转成波长，估计简化最高角分辨率和最大可恢复尺度；说明为什么单一 field/DDID 切片不足以对 mosaic 做最终结论。（5 分）
6. 改变 `npix`、cell size 和网格加权，比较 dirty image 和 PSF 的旁瓣、图像尺度和中心峰值；写出一条可复现的参数选择规则。（5 分）
7. 根据 `DATA` 单位、SOURCE 通量列、SYSCAL 的 `TSYS/JYPERK` 和 HISTORY 校准开关，逐项说明哪些只证明上游校正痕迹、哪些才能建立绝对标度；再列出升级为可发布绝对校准和谱线成像仍须补齐的证据。（4 分）

这个案例已经将第 8 章的增益求解器接到真实 Measurement Set 列，同时保留工程和科学边界：格式真实、数值真实和可解决完整科学问题是三件不同的事。

***
